<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_06_model_training/seq2one/stage_06_02_ridge_seq2one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_06_02 - SEQ2ONE - Modelo Ridge**

**Introducción**

Luego de establecer el baseline Naive, se introduce **Ridge Regression** como
primer modelo **entrenable** del stage_07 bajo un enfoque **seq2one**.

Ridge es una regresión lineal con **regularización L2**, diseñada para manejar
vectores de alta dimensión y features correlacionadas, manteniendo un control
explícito de la complejidad del modelo.

En este proyecto, Ridge permite evaluar cuánto valor puede capturarse mediante
una relación lineal directa entre la ventana histórica intradía
(60 x 20 → 1200 features) y el target escalar futuro.

---

**Rol en el pipeline**

- Primer modelo con capacidad de aprendizaje real.
- Referencia lineal fuerte y estable.
- Punto de comparación obligatorio para modelos no lineales posteriores.
- Si modelos más complejos no superan a Ridge en VALID, su aporte es cuestionable.

---

**Esquema general**

- **Entrada (X):** vector aplanado de 1200 features.
- **Salida (Y):** valor escalar futuro.
- **Entrenamiento:** conjunto TRAIN.
- **Evaluación:** conjunto VALID.
- **Regularización:** L2 (controlada por el parámetro \(\alpha\)).

Las métricas obtenidas se almacenan como artefactos y se utilizan posteriormente en el **stage_08** para la comparación final entre modelos.


## **1. Imports + paths**

In [1]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

## **2. Acceso a drive**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


## **3. Rutas de ventanas seq2one y scalers**

In [3]:
from pathlib import Path
import os

WINDOWS_SEQ2ONE_DIR = Path(
    os.environ.get("WINDOWS_SEQ2ONE_DIR", "data/windows/seq2one/")
)

SCALERS_DIR = Path(
    os.environ.get("SCALERS_DIR", "data/scaled/")
)

window_sizes = [30, 60, 90, 120, 180]
targets = ['delta_60', 'delta_90', 'ret_60', 'ret_90']
splits = ['train', 'valid', 'test']

In [4]:
windows_paths = {}

for w in window_sizes:
    windows_paths[w] = {}

    for t in targets:
        windows_paths[w][t] = {}

        for s in splits:
            path = (
                DRIVE_DIR
                / WINDOWS_SEQ2ONE_DIR
                / f"L{w}"
                / f"windows_{t}_{s}.npz"
            )

            windows_paths[w][t][s] = path

#display(windows_paths)

#Como llamarlo:
#path_train_L60_delta = windows_paths[60]['delta_90']['train']
#print(path_train_L60_delta)

In [5]:
scalers_paths = {}
for t in targets:
  scalers_paths[t] = {}
  path = (
                DRIVE_DIR
                / SCALERS_DIR
                / f"scaler_{t}.pkl"
            )

  scalers_paths[t] = path

display(scalers_paths)

{'delta_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_60.pkl'),
 'delta_90': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_90.pkl'),
 'ret_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_ret_60.pkl'),
 'ret_90': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_ret_90.pkl')}

## **4. Reproducibilidad**

In [6]:
def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

## **5. Importar métricas comunes desde .py**

In [7]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2one_metrics import compute_seq2one_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [8]:
print(compute_seq2one_metrics.__doc__)


    Calcula métricas simples y comparables para modelos seq2one.

    Parámetros
    ----------
    y_true : np.ndarray
        Valores reales con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    y_pred : np.ndarray
        Valores predichos con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    compute_r2 : bool
        Si True, calcula R² sobre el vector completo.
    da_ignore_zeros : bool
        Si True, ignora casos donde el signo sea 0 en y_true o y_pred al calcular DA.
    allow_seq_inputs_take_last : bool
        Si True, permite inputs 2D (n_samples, seq_len) y toma el último paso [:, -1].
        Útil si algún modelo devuelve secuencia pero usted lo evalúa como many-to-one.

    Retorna
    -------
    metrics : dict
        Diccionario con métricas globales.
    


## **5. Carga de ventanas**

In [9]:
# --------------------------------------------------
# Función común: carga .npz estándar (X, y)
# --------------------------------------------------
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.

    Espera claves:
    - 'X': array (n_samples, seq_len, n_features)
    - 'y' o 'Y': array (n_samples, seq_len) o (n_samples, seq_len, 1)
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Carga el NPZ (lectura).
    data = np.load(path)

    # Lee X (obligatoria).
    if "X" not in data:
        raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
    X = data["X"]

    # Lee y: soporta 'y' (convención usada) o 'Y' (por compatibilidad).
    if "y" in data:
        y = data["y"]
    elif "Y" in data:
        y = data["Y"]
    else:
        raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

    # Devuelve X e y.
    return X, y

In [10]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [11]:
from typing import Any, Dict, Mapping
from pathlib import Path

# --------------------------------------------------
# Carga completa: ventanas + scaler por window_size y target
# --------------------------------------------------
def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scalers_path: Mapping[str, Path],
) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler correspondiente
    a un (window_size, target).

    windows_paths[L][target][split] -> Path
    scalers_path[target] -> Path
    """

    # --------------------------
    # 1) Validaciones
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    if target not in scalers_path:
        raise KeyError(f"target='{target}' no existe en scalers_path")

    # --------------------------
    # 2) Paths
    # --------------------------
    train_path = windows_paths[window_size][target]["train"]
    valid_path = windows_paths[window_size][target]["valid"]
    test_path  = windows_paths[window_size][target]["test"]
    scaler_path = scalers_paths[target]

    # --------------------------
    # 3) Carga
    # --------------------------
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test,  y_test  = load_npz_windows(test_path)

    scaler = load_scaler(scaler_path)

    # --------------------------
    # 4) Inferir horizonte
    # --------------------------
    horizon = int(target.split("_")[-1])

    # --------------------------
    # 5) Retorno
    # --------------------------
    return {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {"X": X_train, "y": y_train},
        "valid": {"X": X_valid, "y": y_valid},
        "test":  {"X": X_test,  "y": y_test},
    }

In [12]:
def create_bundles(window_size, targets: list, windows_paths = windows_paths, scalers_paths = scalers_paths ):

  bundle_60 = load_windows_and_scaler(
      window_size=window_size,
      target=targets[0],
      windows_paths=windows_paths,
      scalers_path=scalers_paths,
  )

  bundle_90 = load_windows_and_scaler(
      window_size=window_size,
      target=targets[1],
      windows_paths=windows_paths,
      scalers_path=scalers_paths,
 )

  # --------------------------------------------------
  # Verificación rápida
  # --------------------------------------------------

  # Shapes H60.
  print("H60 Train:", bundle_60["train"]["X"].shape, bundle_60["train"]["y"].shape)
  print("H60 Valid:", bundle_60["valid"]["X"].shape, bundle_60["valid"]["y"].shape)
  print("H60 Test :", bundle_60["test"]["X"].shape,  bundle_60["test"]["y"].shape)

  # Shapes H90.
  print("H90 Train:", bundle_90["train"]["X"].shape, bundle_90["train"]["y"].shape)
  print("H90 Valid:", bundle_90["valid"]["X"].shape, bundle_90["valid"]["y"].shape)
  print("H90 Test :", bundle_90["test"]["X"].shape,  bundle_90["test"]["y"].shape)

  # Información útil (scaler).
  print("Scaler H60:", type(bundle_60["scaler"]).__name__)
  print("Scaler H90:", type(bundle_90["scaler"]).__name__)

  return bundle_60, bundle_90

In [13]:
#bundle_delta_60, bundle_delta_90 = create_bundles(window_size = 30, targets = ['delta_60', 'delta_90'], windows_paths = windows_paths, scalers_paths = scalers_paths)
#bundle_ret_60, bundle_ret_90 = create_bundles(window_size = 30, targets = ['ret_60', 'ret_90'], windows_paths = windows_paths, scalers_paths = scalers_paths)

NOTA IMPORTANTE: COMO ACCEDER A LAS VENTANAS

Para el horizonte: `h`

  - Train
    - `X`: `bundle_h["train"]["X"]`
    - `y`: `bundle_h["train"]["y"]`

  - Valid
    - `X`: `bundle_h["valid"]["X"]`
    - `y`:`bundle_h["valid"]["y"]`

  - Test
    - `X`: `bundle_h["test"]["X"]`
    - `y`: `bundle_h["test"]["y"]`

  - Scaler
    - `bundle_h["scaler"]`

## **6. Sanity Check**

In [14]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_float_array(a: Any, *, name: str) -> np.ndarray:
    """Convierte a np.ndarray float64 y valida finitud."""
    arr = np.asarray(a, dtype=np.float64)
    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}")
    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).
    Acepta: (n,), (n,1). Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y
    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)
    raise ValueError(f"{name} shape inválido para seq2one. Se esperaba (n,) o (n,1). Recibido {y.shape}")


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiera (seq_len, n_features, mode) desde X.
    mode:
      - "3d": X=(n, seq_len, n_features)
      - "2d": X=(n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"
    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"
    raise ValueError(f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}")

In [15]:
# ============================================================
# 2) Sanity check principal (seq2one)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como "d_flat" esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado). Ej: 60*20=1200.
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len) (caso raro), permite tomar y[:, -1].
        Por defecto False (recomendado).
    """
    X = _as_float_array(X, name=f"X[{split_name}]")
    y = _as_float_array(y, name=f"y[{split_name}]")

    # Normalizar y
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        # caso tolerante: y=(n,seq_len) -> tomar último
        y = y[:, -1]
    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # Inferir modo y dimensiones de X
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # Validaciones básicas n_samples
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # Validación de shapes según modo
    if mode == "3d":
        # expected_seq_len / expected_n_features
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, recibido={seq_len}. X.shape={X.shape}"
            )
        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        # Si expected_flat_dim está, valida contra eso
        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, recibido={d_flat}. X.shape={X.shape}"
            )

        # Si no hay expected_flat_dim pero sí expected_seq_len, úselo como d_flat esperado
        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, recibido={d_flat}. X.shape={X.shape}"
            )

        # expected_n_features no aplica en 2D
        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim (ej: 1200) o pase X en 3D."
            )

    # Validación extra: varianza de y (para detectar targets constantes)
    y_std = float(np.std(y))
    if verbose:
        info = {
            "split": split_name,
            "X_shape": tuple(X.shape),
            "y_shape": tuple(y.shape),
            "mode": mode,
            "seq_len": seq_len if mode == "3d" else None,
            "n_features": n_features if mode == "3d" else None,
            "flat_dim": int(X.shape[1]) if mode == "2d" else None,
            "y_mean": float(np.mean(y)),
            "y_std": y_std,
            "y_min": float(np.min(y)),
            "y_max": float(np.max(y)),
        }
        print(
            f"[sanity_check_seq2one] {split_name} | X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | y_std={y_std:.6f}"
        )

    return {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "y_mean": float(np.mean(y)),
        "y_std": float(np.std(y)),
        "y_min": float(np.min(y)),
        "y_max": float(np.max(y)),
    }

In [16]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # Setear esperados desde TRAIN si no se dieron
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        # En 3D no hace falta expected_flat_dim
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        # Para evitar confusión, no usamos expected_seq_len/n_features en 2D
        expected_seq_len = expected_seq_len  # puede quedar None
        expected_n_features = None

    # Ejecutar checks
    out_tr = sanity_check_seq2one(
        X_tr, y_tr, f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_va = sanity_check_seq2one(
        X_va, y_va, f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_te = sanity_check_seq2one(
        X_te, y_te, f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    h = bundle.get("horizon", "NA")
    if verbose:
        print(f"OK {tag} (h={h})")

    return {"train": out_tr, "valid": out_va, "test": out_te, "horizon": h}


def run_sanity_checks_all_horizons_seq2one(
    bundle_60: Dict[str, Any],
    bundle_90: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """Corre sanity checks para ambos horizontes (ej: 60 y 90)."""
    out_60 = run_sanity_checks_for_bundle_seq2one(bundle_60, tag="h60", verbose=verbose)
    out_90 = run_sanity_checks_for_bundle_seq2one(bundle_90, tag="h90", verbose=verbose)
    return {"h60": out_60, "h90": out_90}

In [17]:
#summary_delta = run_sanity_checks_all_horizons_seq2one(bundle_delta_60, bundle_delta_90)
#summary_ret = run_sanity_checks_all_horizons_seq2one(bundle_ret_60, bundle_ret_90)

# **DEFINICIÓN DE MODELO**

## **7. Definición del modelo — placeholder**

### **7.1. Modelo Ridge Regression (seq2one)**

**Idea básica**

**Ridge Regression** es una regresión lineal con **regularización L2**.  
Aprende un vector de pesos **w** que relaciona linealmente la entrada aplanada
con el target escalar, penalizando pesos grandes para reducir el sobreajuste.

Formalmente:

$$
\hat{y}_t = \mathbf{w}^\top \mathbf{x}_t + b
$$

con función objetivo:

$$
\min_{\mathbf{w}, b}
\sum_t (y_t - \hat{y}_t)^2
\;+\;
\alpha \sum_i w_i^2
$$

donde $\alpha\$ controla la **fuerza de la regularización**.

---

**Regularización (Ridge / Lasso)**

- **Ridge (L2):**
  - Penaliza el cuadrado de los coeficientes.
  - Reduce la magnitud de los pesos sin anularlos.
- **Lasso (L1):**
  - Penaliza el valor absoluto de los coeficientes.
  - Puede llevar pesos exactamente a cero (sparsity).

**Riesgo:** Bajo, controlado por diseño.  
La regularización es **intrínseca al modelo** y está gobernada por el
hiperparámetro \(\alpha\).

No se requieren técnicas adicionales como **dropout** o **early stopping**,
ya que no se trata de un modelo iterativo por épocas.

---

**Por qué Ridge encaja bien en este proyecto**

- Entrada de **alta dimensión**: 60 × 20 = **1200 features**.
- Features **altamente correlacionadas** (estructura temporal).
- Modelo:
  - simple,
  - estable,
  - rápido de entrenar,
  - interpretable como referencia lineal.

Ridge actúa como el **baseline entrenable** contra el cual se comparan
modelos más complejos (MLP, LSTM, TCN, Transformer).

---

**Hiperparámetros iniciales**

Para este stage (sin tuning):

- `alpha`: fijo (por ejemplo, `1.0`)
- `fit_intercept`: `True`
- `random_state`: no aplica
- **Sin validación interna** (la evaluación se realiza externamente en VALID)

El ajuste fino del coeficiente de regularización se aborda en etapas posteriores.


### **7.2. Implementación del modelo**

In [18]:
from sklearn.linear_model import Ridge

def build_ridge_model(*, alpha: float = 1.0) -> Ridge:
    """
    Construye un modelo Ridge Regression para seq2one.

    Parámetros
    ----------
    alpha : float
        Fuerza de regularización L2.

    Retorna
    -------
    model : sklearn.linear_model.Ridge
        Modelo Ridge configurado.
    """
    model = Ridge(
        alpha=alpha,
        fit_intercept=True,
    )
    return model


### **7.3. Entrenamiento (TRAIN)**

In [19]:
def train_ridge_for_bundle(bundle, *, alpha: float = 1.0):
    X_train = bundle["train"]["X"]
    y_train = bundle["train"]["y"]

    model = build_ridge_model(alpha=alpha)
    model.fit(X_train, y_train)
    return model

In [20]:
#ridge_60 = train_ridge_for_bundle(bundle_60, alpha=1.0)
#ridge_90 = train_ridge_for_bundle(bundle_90, alpha=1.0)

## **8. Métricas ML**

In [21]:
import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    horizon: int,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte un dict de métricas seq2one en una fila de DataFrame.
    """

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA": metrics.get("DA"),
    }])

In [22]:
def get_metrics (bundle, ridge_model):

    # -------- VALID --------
    X_valid = bundle ['valid']['X']
    y_pred_valid = ridge_model.predict(X_valid)

    # --------TEST --------
    X_test = bundle ['test']['X']
    y_pred_test = ridge_model.predict(X_test)

    y_valid = bundle["valid"]["y"]
    y_test  = bundle["test"]["y"]

    metrics_valid = compute_seq2one_metrics(y_valid, y_pred_valid, compute_r2=True)
    metrics_test  = compute_seq2one_metrics(y_test,  y_pred_test,  compute_r2=True)

    return metrics_valid, metrics_test

## **9. Ejecución completa**

In [23]:
import pandas as pd
import gc

def run_ridge(window_size: int, *, alpha: float = 1.0, verbose: bool = True):

    size = window_size

    if verbose:
        print("\n" + "=" * 80)
        print(f"RIDGE | SEQ2ONE | WINDOW_SIZE=L{size} | alpha={alpha}")
        print("=" * 80)

    # --------------------------------------------------
    # Crear bundles
    # --------------------------------------------------
    if verbose:
        print(f"\n[BUILD] L{size} | targets = ['delta_60', 'delta_90']")

    bundle_delta_60, bundle_delta_90 = create_bundles(
        window_size=size,
        targets=['delta_60', 'delta_90'],
        windows_paths=windows_paths,
        scalers_paths=scalers_paths,
        # splits=("train","valid","test")  # o ("valid","test") si lo soportas
    )

    if verbose:
        print(f"\n[BUILD] L{size} | targets = ['ret_60', 'ret_90']")

    bundle_ret_60, bundle_ret_90 = create_bundles(
        window_size=size,
        targets=['ret_60', 'ret_90'],
        windows_paths=windows_paths,
        scalers_paths=scalers_paths,
    )

    bundles = [bundle_delta_60, bundle_delta_90, bundle_ret_60, bundle_ret_90]

    # --------------------------------------------------
    # Entrenar + métricas (1 modelo por bundle)
    # --------------------------------------------------
    rows = []

    for bundle in bundles:
        if verbose:
            print(f"\n[TRAIN] L{bundle['window_size']} | target={bundle['target']} | ridge alpha={alpha}")

        model = train_ridge_for_bundle(bundle, alpha=alpha)

        metrics_valid, metrics_test = get_metrics(bundle, model)

        # VALID
        rows.append(
            metrics_to_df(
                metrics_valid,
                model="ridge",
                split="valid",
                horizon=bundle["horizon"],
                window_size=bundle["window_size"],
                target=bundle["target"],
            )
        )

        # TEST
        rows.append(
            metrics_to_df(
                metrics_test,
                model="ridge",
                split="test",
                horizon=bundle["horizon"],
                window_size=bundle["window_size"],
                target=bundle["target"],
            )
        )

    df_ridge_metrics = (
        pd.concat(rows, ignore_index=True)
          .sort_values(["window_size", "target", "split", "horizon_min", "model"])
          .reset_index(drop=True)
    )

    if verbose:
        print(f"\n[DONE] L{size} | rows={len(df_ridge_metrics)}")
        print(df_ridge_metrics[["window_size", "target", "split", "horizon_min", "model"]]
              .drop_duplicates()
              .to_string(index=False))

    # --------------------------------------------------
    # Cleanup
    # --------------------------------------------------
    del bundle_delta_60, bundle_delta_90, bundle_ret_60, bundle_ret_90, bundles
    gc.collect()

    return df_ridge_metrics

In [24]:
dfs = []
for size in window_sizes:
    dfs.append(run_ridge(size, alpha=1.0, verbose=True))

df_ridge_all_sizes = pd.concat(dfs, ignore_index=True)
df_ridge_all_sizes



RIDGE | SEQ2ONE | WINDOW_SIZE=L30 | alpha=1.0

[BUILD] L30 | targets = ['delta_60', 'delta_90']
H60 Train: (357504, 30, 36) (357504,)
H60 Valid: (76440, 30, 36) (76440,)
H60 Test : (76832, 30, 36) (76832,)
H90 Train: (357504, 30, 36) (357504,)
H90 Valid: (76440, 30, 36) (76440,)
H90 Test : (76832, 30, 36) (76832,)
Scaler H60: StandardScaler
Scaler H90: StandardScaler

[BUILD] L30 | targets = ['ret_60', 'ret_90']
H60 Train: (357504, 30, 36) (357504,)
H60 Valid: (76440, 30, 36) (76440,)
H60 Test : (76832, 30, 36) (76832,)
H90 Train: (357504, 30, 36) (357504,)
H90 Valid: (76440, 30, 36) (76440,)
H90 Test : (76832, 30, 36) (76832,)
Scaler H60: StandardScaler
Scaler H90: StandardScaler

[TRAIN] L30 | target=delta_60 | ridge alpha=1.0


ValueError: Found array with dim 3. Ridge expected <= 2.

## **10. Guardar artefactos para Stage_08**

In [ ]:
from pathlib import Path
import pandas as pd

def save_seq2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"seq2one_{name}_metrics.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path


In [ ]:
save_seq2one_metrics(
    df_ridge_all_sizes,
    name="ridge",
)